In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
df = pd.read_csv(
    "/content/healthcare_cleaned.csv"
)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully.
Dataset shape: (54966, 22)


,patient_id,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,age_group,admission_year,admission_month,admission_day,admission_quarter,length_of_stay
0,1,Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons And Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal,Young Adult,2024,January,Wednesday,Q1,2
1,2,Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,Senior,2019,August,Tuesday,Q3,6
2,3,Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook Plc,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal,Elderly,2022,September,Thursday,Q3,15
3,4,Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers And Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal,Young Adult,2020,November,Wednesday,Q4,30
4,5,Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal,Adult,2022,September,Monday,Q3,20


In [ ]:
df["date_of_admission"] = pd.to_datetime(
    df["date_of_admission"],
    errors="coerce"
)

df["discharge_date"] = pd.to_datetime(
    df["discharge_date"],
    errors="coerce"
)

print(df[["date_of_admission", "discharge_date"]].dtypes)

date_of_admission    datetime64[ns]
discharge_date       datetime64[ns]
dtype: object


In [ ]:
existing_columns = [
    "patient_id",
    "age_group",
    "admission_year",
    "admission_month",
    "admission_day",
    "admission_quarter",
    "length_of_stay"
]

for column in existing_columns:
    print(column, ":", column in df.columns)

patient_id : True
age_group : True
admission_year : True
admission_month : True
admission_day : True
admission_quarter : True
length_of_stay : True


In [ ]:
stay_bins = [-1, 3, 7, 14, np.inf]

stay_labels = [
    "Short Stay",
    "Medium Stay",
    "Long Stay",
    "Extended Stay"
]

df["length_of_stay_category"] = pd.cut(
    df["length_of_stay"],
    bins=stay_bins,
    labels=stay_labels
)

In [ ]:
df["length_of_stay_category"].value_counts(
    dropna=False
)

,count
length_of_stay_category,
Extended Stay,29201
Long Stay,12879
Medium Stay,7420
Short Stay,5466


In [ ]:
age_band_bins = [
    0,
    18,
    30,
    45,
    60,
    75,
    np.inf
]

age_band_labels = [
    "0-18",
    "19-30",
    "31-45",
    "46-60",
    "61-75",
    "76+"
]

df["age_band"] = pd.cut(
    df["age"],
    bins=age_band_bins,
    labels=age_band_labels,
    include_lowest=True
)

In [ ]:
df["age_band"].value_counts(
    dropna=False
).sort_index()

,count
age_band,
0-18,886
19-30,9519
31-45,12116
46-60,12273
61-75,12098
76+,8074


In [ ]:
df["admission_month_number"] = (
    df["date_of_admission"].dt.month
)

In [ ]:
df[
    [
        "date_of_admission",
        "admission_month",
        "admission_month_number"
    ]
].head()

,date_of_admission,admission_month,admission_month_number
0,2024-01-31,January,1
1,2019-08-20,August,8
2,2022-09-22,September,9
3,2020-11-18,November,11
4,2022-09-19,September,9


In [ ]:
def assign_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Autumn"

In [ ]:
df["admission_season"] = (
    df["admission_month_number"]
    .apply(assign_season)
)

In [ ]:
df["admission_season"].value_counts()

,count
admission_season,
Monsoon,18708
Summer,13655
Winter,13482
Autumn,9121


In [ ]:
df["admission_day_type"] = np.where(
    df["date_of_admission"].dt.dayofweek >= 5,
    "Weekend",
    "Weekday"
)

In [ ]:
df["admission_day_type"].value_counts()

,count
admission_day_type,
Weekday,39294
Weekend,15672


In [ ]:
df["billing_category"] = pd.qcut(
    df["billing_amount"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ],
    duplicates="drop"
)

In [ ]:
df["billing_category"].value_counts()

,count
billing_category,
Medium,13794
Low,13742
Very High,13742
High,13688


In [ ]:
df["billable_days"] = (
    df["length_of_stay"]
    .clip(lower=1)
)

In [ ]:
print(
    "Zero-day stays:",
    (df["length_of_stay"] == 0).sum()
)

Zero-day stays: 0


In [ ]:
df["billing_per_day"] = (
    df["billing_amount"] /
    df["billable_days"]
).round(2)

In [ ]:
df["billing_per_day"].describe()

,billing_per_day
count,54966.000000
mean,3398.445303
std,5798.072153
min,0.950000
25%,860.875000
50%,1652.750000
75%,3210.917500
max,52211.850000


In [ ]:
print(
    "Infinite values:",
    np.isinf(df["billing_per_day"]).sum()
)

Infinite values: 0


In [ ]:
df["insurance_provider"].unique()

array(['Blue Cross', 'Medicare', 'Aetna', 'UnitedHealthcare', 'Cigna'],
      dtype=object)

In [ ]:
private_providers = [
    "Aetna",
    "Blue Cross",
    "Cigna",
    "UnitedHealthcare"
]

df["insurance_type"] = np.where(
    df["insurance_provider"].isin(private_providers),
    "Private",
    "Government"
)

In [ ]:
pd.crosstab(
    df["insurance_provider"],
    df["insurance_type"]
)

insurance_type,Government,Private
insurance_provider,,
Aetna,0,10822
Blue Cross,0,10952
Cigna,0,11139
Medicare,11039,0
UnitedHealthcare,0,11014


In [ ]:
df["long_stay_flag"] = np.where(
    df["length_of_stay"] > 7,
    "Yes",
    "No"
)

In [ ]:
df["long_stay_flag"].value_counts()

,count
long_stay_flag,
Yes,42080
No,12886


In [ ]:
# High-billing flag : Use the 75th percentile as a data-driven threshold.
high_billing_threshold = (
    df["billing_amount"].quantile(0.75)
)

print(
    "High billing threshold:",
    round(high_billing_threshold, 2)
)

High billing threshold: 37819.86


In [ ]:
df["high_billing_flag"] = np.where(
    df["billing_amount"] >= high_billing_threshold,
    "Yes",
    "No"
)

In [ ]:
df["high_billing_flag"].value_counts()

,count
high_billing_flag,
No,41224
Yes,13742


In [ ]:
df["analytical_risk_segment"] = np.select(
    [
        (
            (df["age"] >= 65) &
            (df["test_results"] == "Abnormal")
        ),
        (
            (df["age"] >= 65) |
            (df["test_results"] == "Abnormal") |
            (
                df["medical_condition"]
                .isin(["Cancer", "Hypertension"])
            )
        )
    ],
    [
        "Higher Priority",
        "Review Recommended"
    ],
    default="Standard"
)

In [ ]:
df["analytical_risk_segment"].value_counts()

,count
analytical_risk_segment,
Review Recommended,32468
Standard,16827
Higher Priority,5671


In [ ]:
outcome_mapping = {
    "Normal": "Routine Monitoring",
    "Abnormal": "Review Recommended",
    "Inconclusive": "Further Evaluation"
}

df["patient_outcome_label"] = (
    df["test_results"]
    .map(outcome_mapping)
)

In [ ]:
pd.crosstab(
    df["test_results"],
    df["patient_outcome_label"]
)

patient_outcome_label,Further Evaluation,Review Recommended,Routine Monitoring
test_results,,,
Abnormal,0,18437,0
Inconclusive,18198,0,0
Normal,0,0,18331


In [ ]:
# Stay cost intensity
billing_per_day_median = (
    df["billing_per_day"].median()
)

df["stay_cost_intensity"] = np.where(
    df["billing_per_day"] >= billing_per_day_median,
    "Above Median",
    "Below Median"
)

In [ ]:
df["stay_cost_intensity"].value_counts()

,count
stay_cost_intensity,
Above Median,27483
Below Median,27483


In [ ]:
new_features = [
    "length_of_stay_category",
    "age_band",
    "admission_month_number",
    "admission_season",
    "admission_day_type",
    "billing_category",
    "billable_days",
    "billing_per_day",
    "insurance_type",
    "long_stay_flag",
    "high_billing_flag",
    "analytical_risk_segment",
    "patient_outcome_label",
    "stay_cost_intensity"
]

df[new_features].head()

,length_of_stay_category,age_band,admission_month_number,admission_season,admission_day_type,billing_category,billable_days,billing_per_day,insurance_type,long_stay_flag,high_billing_flag,analytical_risk_segment,patient_outcome_label,stay_cost_intensity
0,Short Stay,19-30,1,Winter,Weekday,Medium,2,9428.14,Private,No,No,Review Recommended,Routine Monitoring,Above Median
1,Medium Stay,61-75,8,Monsoon,Weekday,High,6,5607.22,Government,No,No,Standard,Further Evaluation,Above Median
2,Extended Stay,76+,9,Monsoon,Weekday,High,15,1863.67,Private,Yes,No,Review Recommended,Routine Monitoring,Above Median
3,Extended Stay,19-30,11,Autumn,Weekday,Very High,30,1263.66,Government,Yes,Yes,Review Recommended,Review Recommended,Below Median
4,Extended Stay,31-45,9,Monsoon,Weekday,Medium,20,711.92,Private,Yes,No,Review Recommended,Review Recommended,Below Median


In [ ]:
print("Final shape:", df.shape)

print(
    "Duplicate records:",
    df.duplicated().sum()
)

print(
    "Negative billing values:",
    (df["billing_amount"] < 0).sum()
)

print(
    "Negative length-of-stay values:",
    (df["length_of_stay"] < 0).sum()
)

print(
    "Infinite billing-per-day values:",
    np.isinf(df["billing_per_day"]).sum()
)

print("\nMissing values in new features:")

print(
    df[new_features]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

Final shape: (54966, 36)
Duplicate records: 0
Negative billing values: 0
Negative length-of-stay values: 0
Infinite billing-per-day values: 0

Missing values in new features:
length_of_stay_category    0
age_band                   0
admission_month_number     0
admission_season           0
admission_day_type         0
billing_category           0
billable_days              0
billing_per_day            0
insurance_type             0
long_stay_flag             0
high_billing_flag          0
analytical_risk_segment    0
patient_outcome_label      0
stay_cost_intensity        0
dtype: int64


In [ ]:
output_file = (
    "/content/"
    "healthcare_feature_engineered.csv"
)

df.to_csv(
    output_file,
    index=False
)

print("Dataset saved successfully.")
print("Output path:", output_file)
print("Final shape:", df.shape)

Dataset saved successfully.
Output path: /content/healthcare_feature_engineered.csv
Final shape: (54966, 36)


In [ ]:
feature_summary = pd.DataFrame({
    "feature_name": new_features,
    "data_type": [
        str(df[column].dtype)
        for column in new_features
    ],
    "unique_values": [
        df[column].nunique()
        for column in new_features
    ],
    "missing_values": [
        df[column].isnull().sum()
        for column in new_features
    ]
})

feature_summary

,feature_name,data_type,unique_values,missing_values
0,length_of_stay_category,category,4,0
1,age_band,category,6,0
2,admission_month_number,int32,12,0
3,admission_season,object,4,0
4,admission_day_type,object,2,0
5,billing_category,category,4,0
6,billable_days,int64,30,0
7,billing_per_day,float64,47514,0
8,insurance_type,object,2,0
9,long_stay_flag,object,2,0


In [ ]:
feature_summary.to_csv(
    "/content/feature_engineering_summary.csv",
    index=False
)